# AIML Assignment 2 – Problem Solving Through Search

**Name:** Shivat Dhar  
**PRN:** 202401080003  
**Batch:** A3

## Common Graph and Heuristic Setup

In [ ]:
graph = {
    "S": ["A", "B"],
    "A": ["B", "C"],
    "B": ["C", "G"],
    "C": ["G"],
    "G": []
}

weighted_graph = {
    "S": [("A", 2), ("B", 5)],
    "A": [("B", 1), ("C", 6)],
    "B": [("C", 2), ("G", 8)],
    "C": [("G", 3)],
    "G": []
}

heuristic = {"S": 7, "A": 5, "B": 4, "C": 2, "G": 0}
START, GOAL = "S", "G"

## 2. Uninformed Search – Breadth First Search (BFS)

In [ ]:
from collections import deque

def bfs(graph, start, goal):
    queue = deque([(start, [start])])
    visited = {start}
    expanded = 0

    while queue:
        node, path = queue.popleft()
        expanded += 1

        if node == goal:
            return path, expanded

        for neighbor in graph[node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append((neighbor, path + [neighbor]))

    return None, expanded

## 3. Uninformed Search – Depth First Search (DFS)

In [ ]:
def dfs(graph, start, goal):
    stack = [(start, [start])]
    visited = set()
    expanded = 0

    while stack:
        node, path = stack.pop()

        if node in visited:
            continue

        visited.add(node)
        expanded += 1

        if node == goal:
            return path, expanded

        for neighbor in reversed(graph[node]):
            if neighbor not in visited:
                stack.append((neighbor, path + [neighbor]))

    return None, expanded

## 4. Uninformed Search – Uniform Cost Search (UCS)

In [ ]:
import heapq

def uniform_cost_search(graph, start, goal):
    pq = [(0, start, [start])]
    best_cost = {start: 0}
    expanded = 0

    while pq:
        cost, node, path = heapq.heappop(pq)

        if cost != best_cost.get(node):
            continue

        expanded += 1

        if node == goal:
            return path, cost, expanded

        for neighbor, edge_cost in graph[node]:
            new_cost = cost + edge_cost

            if new_cost < best_cost.get(neighbor, float("inf")):
                best_cost[neighbor] = new_cost
                heapq.heappush(
                    pq, (new_cost, neighbor, path + [neighbor])
                )

    return None, float("inf"), expanded

## 5. Informed Search – Greedy Best-First Search

In [ ]:
import heapq

def greedy_best_first(graph, start, goal, heuristic):
    pq = [(heuristic[start], start, [start])]
    visited = set()
    expanded = 0

    while pq:
        _, node, path = heapq.heappop(pq)

        if node in visited:
            continue

        visited.add(node)
        expanded += 1

        if node == goal:
            return path, expanded

        for neighbor in graph[node]:
            if neighbor not in visited:
                heapq.heappush(
                    pq, (heuristic[neighbor], neighbor,
                         path + [neighbor])
                )

    return None, expanded

## 6. Informed Search – A* Search

In [ ]:
def a_star(graph, start, goal, heuristic):
    import heapq

    pq = [(heuristic[start], 0, start, [start])]
    best_g = {start: 0}
    expanded = 0

    while pq:
        f, g, node, path = heapq.heappop(pq)

        if g != best_g.get(node):
            continue

        expanded += 1

        if node == goal:
            return path, g, expanded

        for neighbor, edge_cost in graph[node]:
            new_g = g + edge_cost

            if new_g < best_g.get(neighbor, float("inf")):
                best_g[neighbor] = new_g
                new_f = new_g + heuristic[neighbor]
                heapq.heappush(
                    pq, (new_f, new_g, neighbor,
                         path + [neighbor])
                )

    return None, float("inf"), expanded

## 7. Local Search – Hill Climbing (8-Queens)

In [ ]:
def conflicts(state):
    total = 0
    for i in range(len(state)):
        for j in range(i + 1, len(state)):
            same_row = state[i] == state[j]
            same_diagonal = abs(state[i] - state[j]) == abs(i - j)

            if same_row or same_diagonal:
                total += 1
    return total

def hill_climbing(n=8, seed=42):
    import random
    random.seed(seed)

    state = [random.randrange(n) for _ in range(n)]
    evaluated = 0

    while True:
        current = conflicts(state)
        neighbors = []

        for col in range(n):
            for row in range(n):
                if row != state[col]:
                    next_state = state[:]
                    next_state[col] = row
                    neighbors.append(next_state)

        evaluated += len(neighbors)
        best_state = min(neighbors, key=conflicts)

        if conflicts(best_state) >= current:
            return state, current, evaluated

        state = best_state

## 8. Local Search – Simulated Annealing (8-Queens)

In [ ]:
def simulated_annealing(n=8, steps=10000, seed=42):
    import random
    import math

    random.seed(seed)
    state = [random.randrange(n) for _ in range(n)]
    current = conflicts(state)

    best_state = state[:]
    best_score = current

    for step in range(steps):
        col = random.randrange(n)
        row = random.randrange(n)

        next_state = state[:]
        next_state[col] = row

        new_score = conflicts(next_state)
        temperature = max(10 * (0.995 ** step), 1e-9)
        delta = new_score - current

        if delta < 0 or random.random() < math.exp(-delta / temperature):
            state = next_state
            current = new_score

        if current < best_score:
            best_state = state[:]
            best_score = current

        if best_score == 0:
            break

    return best_state, best_score, step + 1

## 9. Constraint Satisfaction – Backtracking (4-Queens)

In [ ]:
def solve_4_queens():
    n = 4
    assignment = [-1] * n
    assignments_tested = 0

    def safe(row, col):
        for previous_col in range(col):
            previous_row = assignment[previous_col]

            if previous_row == row:
                return False

            if abs(previous_row - row) == abs(previous_col - col):
                return False

        return True

    def backtrack(col):
        nonlocal assignments_tested

        if col == n:
            return True

        for row in range(n):
            assignments_tested += 1

            if safe(row, col):
                assignment[col] = row

                if backtrack(col + 1):
                    return True

                assignment[col] = -1

        return False

    solved = backtrack(0)
    return assignment, assignments_tested, solved

## 10. Run All Algorithms

In [ ]:
print("BFS:", bfs(graph, START, GOAL))
print("DFS:", dfs(graph, START, GOAL))
print("UCS:", uniform_cost_search(weighted_graph, START, GOAL))
print("Greedy:", greedy_best_first(graph, START, GOAL, heuristic))
print("A*:", a_star(weighted_graph, START, GOAL, heuristic))

print("Hill Climbing:", hill_climbing())
print("Simulated Annealing:", simulated_annealing())
print("4-Queens CSP:", solve_4_queens())

## 11. Performance Comparison

In [ ]:
import time
import pandas as pd

def measure(name, function):
    start_time = time.perf_counter()
    result = function()
    elapsed = time.perf_counter() - start_time
    return name, result, elapsed

rows = []

for name, function in [
    ("BFS", lambda: bfs(graph, START, GOAL)),
    ("DFS", lambda: dfs(graph, START, GOAL)),
    ("UCS", lambda: uniform_cost_search(weighted_graph, START, GOAL)),
    ("Greedy", lambda: greedy_best_first(graph, START, GOAL, heuristic)),
    ("A*", lambda: a_star(weighted_graph, START, GOAL, heuristic))
]:
    algorithm, result, elapsed = measure(name, function)

    if name in ("UCS", "A*"):
        path, cost, expanded = result
    else:
        path, expanded = result
        cost = "-"

    rows.append([algorithm, path, cost, expanded, elapsed])

performance_df = pd.DataFrame(
    rows,
    columns=["Algorithm", "Path", "Cost",
             "Nodes Expanded", "Time (seconds)"]
)

performance_df